### 01 — pybaseball_installation_check

Purpose: Confirm `pybaseball` can pull Statcast and Fangraph data end-to-end against this environment.

Caching is enabled and pointed at `../data/cache/` so re-running this notebook is instant after the first pull.

In [ ]:
from pathlib import Path

import pandas as pd
import pybaseball
from pybaseball import cache, statcast

DATA_DIR = Path("..") / "data" / "cache"
DATA_DIR.mkdir(parents=True, exist_ok=True)
cache.config.cache_directory = str(DATA_DIR.resolve())
cache.config.save()
cache.enable()

print("pybaseball:", pybaseball.__version__)
print("pandas    :", pd.__version__)
print("cache dir :", cache.config.cache_directory)

In [ ]:
df = statcast("2024-06-15", "2024-06-15")
df.shape

In [ ]:
print("rows, cols:", df.shape)
print("\ndtype counts:")
print(df.dtypes.value_counts())
df.head(3)

In [ ]:
print(f"{len(df.columns)} columns:")
for c in df.columns:
    print(f"  {c}")

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss[miss > 0].to_frame("pct_missing")

In [ ]:
print("pitch_type missingness:", df["pitch_type"].isna().mean())
print("pitch_name missingness:", df["pitch_name"].isna().mean())
print("\npitch_type value counts:")
print(df["pitch_type"].value_counts(dropna=False))
print("\npitch_name value counts:")
print(df["pitch_name"].value_counts(dropna=False))

---
### Player-Specific Feature Enrichment

The goal is to enrich each pitch row with **prior-year** pitcher and batter arsenal stats (usage + per-pitch-type outcomes).

| Table | pybaseball function | Why |
|---|---|---|
| Pitcher arsenal | `statcast_pitcher_arsenal_stats(year)` | `pitch_usage` + outcome stats per pitch type |
| Batter vs pitch types | `statcast_batter_pitch_arsenal(year)` | `bat_pitch_usage` + outcome stats per pitch type |

Enrichment is applied via shared `enrich_statcast()` from `utils.enrichment`.

In [ ]:
### Pitcher arsenal stats (prior year) — includes pitch_usage
from pybaseball import statcast_pitcher_arsenal_stats

SEASON = int(df["game_year"].max())
PRIOR  = SEASON - 1

pit_outcomes = statcast_pitcher_arsenal_stats(PRIOR, minPA=25)
print("pit_outcomes shape:", pit_outcomes.shape)
print("\nColumns:", pit_outcomes.columns.tolist())
pit_outcomes[pit_outcomes["pitch_type"] == "FF"][["player_id", "pitch_type", "pitch_usage", "whiff_percent"]].head(3)

In [ ]:
### Batter vs pitch-type stats (prior year)
from pybaseball import statcast_batter_pitch_arsenal

bat_vs_pitch = statcast_batter_pitch_arsenal(PRIOR, minPA=25)
print("bat_vs_pitch shape:", bat_vs_pitch.shape)
print("\nColumns:", bat_vs_pitch.columns.tolist())
bat_vs_pitch[bat_vs_pitch["pitch_type"] == "FF"][["player_id", "pitch_type", "pitch_usage", "whiff_percent"]].head(3)

In [ ]:
### End-to-end merge via enrich_statcast()
import sys
sys.path.append(str(Path("..").resolve()))

from utils.enrichment import enrich_statcast

df_enr = enrich_statcast(df, PRIOR)

print(f"Raw shape     : {df.shape}")
print(f"Enriched shape: {df_enr.shape}")
if "pitch_usage_FF" in df_enr.columns:
    print(f"Pitcher usage coverage (pitch_usage_FF non-null): {df_enr['pitch_usage_FF'].notna().mean():.1%}")

show_cols = (
    ["pitcher", "p_throws", "pitch_usage_FF", "pitch_usage_SL", "pitch_usage_CH",
     "batter", "stand", "bat_pitch_usage_FF"]
)
show_cols = [c for c in show_cols if c in df_enr.columns]
df_enr[show_cols].dropna(subset=["pitch_usage_FF"]).head(3)